# 10 — Broadcasting and Vectorization

**Master syllabus coverage:** V2 1.2, 2.1, 2.3

## Why this matters

Vectorized tensor algebra is how mathematical intent reaches fast kernels. Broadcasting and layout knowledge prevent silent shape errors and unnecessary memory copies.

## Learning objectives

- Derive broadcasted output shapes from right to left.
- Translate nested loops into matrix, batched, and Einstein notation.
- Explain views, copies, strides, expansion, and contiguity.
- Design a modest benchmark without drawing conclusions from one noisy timing.

Work through the notebook in order. Predict shapes and results before running each code cell. An assertion failure is part of the lesson: read it before changing code.


## 1. Broadcasting is alignment from the right

Compare dimensions from right to left. Two dimensions are compatible when they are
equal or either one is 1; missing leading dimensions act like 1. Broadcasting creates
a logical expanded view rather than copying values in most cases.

Example: `[B,T,C] + [C] → [B,T,C]`. A feature bias is reused for every token in every
batch item. But `[B,T,C] + [T]` fails because `T` is compared with `C`.


In [ ]:
import time
import numpy as np
import torch

torch.manual_seed(42)
B, T, C = 2, 3, 4
x = torch.arange(B * T * C, dtype=torch.float32).reshape(B, T, C)
feature_bias = torch.tensor([0.1, 0.2, 0.3, 0.4])  # [C]
token_bias = torch.tensor([1.0, 2.0, 3.0]).view(1, T, 1)  # [1,T,1]

y = x + feature_bias + token_bias
assert y.shape == (B, T, C)
print(y[0])


## 2. Replace loops, preserve meaning

Vectorization expresses a whole operation in array algebra so optimized native kernels
can process it. First write the obvious loop, establish a correct reference, then
vectorize and compare. Faster wrong code is still wrong.


In [ ]:
rng = np.random.default_rng(42)
inputs = rng.normal(size=(32, 64)).astype(np.float32)   # [B,Cin]
weights = rng.normal(size=(128, 64)).astype(np.float32) # [Cout,Cin]
bias = rng.normal(size=(128,)).astype(np.float32)       # [Cout]

loop_output = np.empty((32, 128), dtype=np.float32)
for b in range(inputs.shape[0]):
    for o in range(weights.shape[0]):
        total = bias[o]
        for i in range(weights.shape[1]):
            total += inputs[b, i] * weights[o, i]
        loop_output[b, o] = total

vectorized_output = inputs @ weights.T + bias
np.testing.assert_allclose(loop_output, vectorized_output, rtol=2e-5, atol=2e-5)
print("loop and vectorized affine transforms agree")


## 3. Views, copies, strides, and contiguity

A tensor is storage plus shape, stride, and offset. `transpose` usually changes strides
without moving data. `view` requires a compatible contiguous layout; `reshape` may
return a view or allocate a copy. `contiguous()` explicitly materializes standard layout.


In [ ]:
base = torch.arange(12).reshape(3, 4)
transposed = base.T
contiguous = transposed.contiguous()

print("base stride:", base.stride(), "contiguous:", base.is_contiguous())
print("transpose stride:", transposed.stride(), "contiguous:", transposed.is_contiguous())
print("copied stride:", contiguous.stride(), "contiguous:", contiguous.is_contiguous())
assert base.untyped_storage().data_ptr() == transposed.untyped_storage().data_ptr()
assert contiguous.untyped_storage().data_ptr() != transposed.untyped_storage().data_ptr()


## 4. Batched matrix multiplication and Einstein notation

Attention is dominated by batched matrix products. `einsum` names axis relationships:

$$Y_{bto}=\sum_c X_{btc}W_{oc}$$

The string `btc,oc->bto` is both executable and a compact shape derivation.


In [ ]:
x = torch.randn(2, 5, 4)     # [B,T,C]
w = torch.randn(7, 4)        # [O,C]
via_matmul = x @ w.T         # [B,T,O]
via_einsum = torch.einsum("btc,oc->bto", x, w)
torch.testing.assert_close(via_matmul, via_einsum)

expanded = feature_bias.expand(B, T, C)
print("expanded stride:", expanded.stride(), "storage elements:", expanded.untyped_storage().nbytes() // 4)
assert expanded.stride()[0] == 0 and expanded.stride()[1] == 0


## 5. A careful microbenchmark

Timings are noisy and device operations can be asynchronous. Warm up first, repeat,
synchronize accelerators, and compare equivalent work. This tiny CPU timing is only a
demonstration—not a publishable benchmark.


In [ ]:
a = np.random.default_rng(0).normal(size=(200_000,)).astype(np.float32)
b = np.random.default_rng(1).normal(size=(200_000,)).astype(np.float32)

start = time.perf_counter()
loop_sum = sum(float(left * right) for left, right in zip(a, b))
loop_seconds = time.perf_counter() - start
start = time.perf_counter()
vector_sum = float(a @ b)
vector_seconds = time.perf_counter() - start

print(f"loop={loop_seconds:.4f}s vectorized={vector_seconds:.6f}s")
np.testing.assert_allclose(loop_sum, vector_sum, rtol=2e-5)


## Failure modes to recognize

- **Accidental broadcasting:** compatible shapes produce the wrong semantic pairing.
- **Hidden copy:** a reshape or contiguous conversion increases peak memory.
- **Non-contiguous view:** `view` fails or a kernel takes a slower layout path.
- **Bad benchmark:** startup, asynchronous execution, or unequal work dominates timing.

These are diagnostic patterns, not trivia. When a future model fails, reduce the problem to the smallest example in this notebook and test the relevant invariant.


## Deliberate practice

1. Derive and test `[B,T,1] * [C]`, `[B,1,C] + [T,1]`, and one intentionally invalid pair.
2. Implement batched cosine similarity first with loops and then without Python loops.
3. Transpose a four-axis tensor, inspect its stride, and predict whether `view(-1)` succeeds.

Do not merely rerun the provided cells. Make a prediction, change one variable, and write down why the result did or did not match your prediction.

## Exit ticket

**You are ready to continue when:** you can vectorize a loop, prove numerical equivalence, and explain the output shape and storage behavior.

**Next:** 11 — Vectors and geometric intuition.
